[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JulesMalin/isba2411-nlp/blob/main/Week%205/L9_Demo_SteamReviewAssistant.ipynb)

# ISBA 2411 · Week 5 · Lecture 9
## Should You Play It? — a Steam Game-Review Assistant

Last week you *built* a neural network from scratch. Tonight you *borrow* pretrained ones off **Hugging Face** and point them at thousands of real **Steam game reviews** — no training, three lines each.

Two tools, both new tonight:
- **Ask the reviews** — *question answering*: “how hard is Sekiro?” → it extracts the answer from real reviews.
- **Review Router** — *zero-shot classification*: sort any review into categories **you invent on the spot**.

Then we put them in a live **chat app** the whole class uses — and spend the demo trying to *break* it, because the skill isn't calling the model, it's **judging** it.

---
**Runtime:** Colab (free CPU works; GPU is faster for the Router). Run top to bottom.

## 0 · Setup

In [ ]:
%pip install -q "transformers>=4.40,<5" "gradio>=4.0" scikit-learn pandas

import pandas as pd, re, html, textwrap
from transformers import pipeline
print('setup complete')

## 1 · Load the Steam reviews
~5,000 English reviews across 16 popular games. Each row has the review text plus the player's **thumbs-up/down** (`recommended`) and `hours_played` — real signal we'll use later.

In [ ]:
DATA_URL = 'https://raw.githubusercontent.com/JulesMalin/isba2411-nlp/main/Week%205/steam_reviews_demo.csv'

df = pd.read_csv(DATA_URL)
df['review'] = df['review'].map(lambda t: re.sub(r'\s+', ' ', html.unescape(str(t))).strip())
df = df[df['review'].str.len() > 40].reset_index(drop=True)
print(f'{len(df):,} reviews | {df.game.nunique()} games')
df.sample(3)

## 2 · Warm-up — sentiment in three lines (a Week-4 callback)
You met sentiment in Week 4. Here it is the *easy* way — a Hugging Face `pipeline`. One line to load, one to run. We won't dwell; it's the on-ramp to the two new tools.

In [ ]:
sentiment = pipeline('sentiment-analysis')
for r in df['review'].sample(3, random_state=1):
    o = sentiment(r[:512])[0]
    print(f"{o['label']:8} {o['score']:.2f}  |  {textwrap.shorten(r, 90)}")

## 3 · Tool A — Review Router (zero-shot classification)
**No training, no fixed categories.** You hand the model a review *and a list of labels you invent*, and it scores each one. Under the hood it asks, for every label, *“does this review imply it's about ___?”* — the entailment probability is the score.

That's the magic: **one pretrained model, infinite label sets.**

In [ ]:
router = pipeline('zero-shot-classification', model='facebook/bart-large-mnli')

review = 'Gorgeous game but it ran like garbage on my PC for the first month. Fixed now, worth it.'
labels = ['graphics', 'performance', 'price or value', 'difficulty', 'story']
r = router(review, candidate_labels=labels, multi_label=True)
for lab, sc in zip(r['labels'], r['scores']):
    print(f"{lab:16} {'#'*int(sc*20):20} {sc:.2f}")

### The same review, a different question
Change the labels and you change the *question you're asking the data.* Same text, new lens:

In [ ]:
for lab in [['a complaint','a recommendation','a bug report'],
            ['about the price','about the graphics','about the story']]:
    r = router(review, candidate_labels=lab, multi_label=False)
    print(lab, '->', r['labels'][0], f"({r['scores'][0]:.2f})")

## 4 · Tool B — Ask the reviews (extractive QA)
Given a **question** + some **review text** as context, the model *points to the answer span in the text.* It extracts, it doesn't invent (no hallucination) — but it can only return words already there.

In [ ]:
qa = pipeline('question-answering', model='distilbert-base-cased-distilled-squad')

context = ('Elden Ring is brutal but fair. The open world is stunning, though the '
           'difficulty spikes and some late bosses feel unfair.')
for q in ['How hard is the game?', 'What do people like about it?']:
    a = qa(question=q, context=context)
    print(f"Q: {q}\n   A: {a['answer']!r}  (score {a['score']:.2f})\n")

### Give QA the *right* reviews to read (Week-4 callback)
We reuse **TF-IDF retrieval** to pull the reviews most relevant to the question — and if it names a game we have, we search *within that game*. Keyword search picks *what to read*; QA does *the reading*.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

vec = TfidfVectorizer(stop_words='english', max_features=6000)
M = vec.fit_transform(df['review'])
GAMES = sorted(df['game'].unique())
ALIASES = {'bg3':"Baldur's Gate 3",'baldur':"Baldur's Gate 3",'gta':'Grand Theft Auto V',
           'rdr2':'Red Dead Redemption 2','red dead':'Red Dead Redemption 2','cs2':'Counter-Strike 2',
           'counter strike':'Counter-Strike 2','tf2':'Team Fortress 2','witcher':'The Witcher 3',
           'doom':'DOOM Eternal','stardew':'Stardew Valley','no mans sky':"No Man's Sky"}

def detect_game(text):
    t = text.lower()
    for g in GAMES:
        if g.lower() in t: return g
    for k, v in ALIASES.items():
        if k in t: return v
    return None

def top_reviews(query, k=3, game=None):
    idx = df.index[df['game'] == game] if game else df.index
    sims = linear_kernel(vec.transform([query]), M[idx]).ravel()
    return df.loc[idx[sims.argsort()[::-1][:k]], 'review'].tolist()

print(top_reviews('how hard is it', k=1, game='Sekiro')[0][:200])

## 5 · Wire both tools into one assistant
A **question** → find the game, retrieve reviews, run QA, and add a **verdict** from the thumbs-up labels. Anything else → tag it with the **Router**. Responses come back as styled cards — the same HTML the chat app renders.

In [ ]:
from IPython.display import HTML, display

QUESTION_STARTS = {'what','why','how','is','are','does','do','can','should','which','when','who','worth'}
DEFAULT_LABELS  = 'difficulty, story, graphics, price or value, bugs or performance, multiplayer'
BAR_COLORS = ['#57b0e6', '#a6cf46', '#c98bdc', '#e0a24e', '#5f7183', '#5f7183']

def looks_like_question(t):
    t = t.strip().lower()
    return t.endswith('?') or (t.split() and t.split()[0] in QUESTION_STARTS)

def _card(inner):
    return ('<div style="background:#151d27;border:1px solid #243040;border-radius:13px;'
            'padding:13px 15px;font-family:system-ui,sans-serif;max-width:540px;">' + inner + '</div>')

def _meter(name, score, color):
    pct = int(round(score * 100))
    return ('<div style="display:flex;align-items:center;gap:9px;margin-top:11px;">'
            f'<span style="font-size:11.5px;color:#93a4b7;">{name}</span>'
            '<span style="flex:1;height:7px;background:#0e1620;border:1px solid #243040;'
            'border-radius:99px;display:block;overflow:hidden;">'
            f'<span style="display:block;height:100%;width:{pct}%;background:{color};"></span></span>'
            f'<span style="font-family:monospace;font-size:11.5px;color:#93a4b7;">{score:.2f}</span></div>')

def _qa_html(game, answer, score):
    verdict = ''
    if game is not None:
        g = df[df['game'] == game]
        pct = int(round((g['recommended'] == 'Recommended').mean() * 100))
        verdict = ('<div style="display:flex;align-items:center;gap:11px;padding:10px 12px;'
                   'border-radius:10px;background:#20320c;border:1px solid #33500f;margin-bottom:11px;">'
                   f'<span style="font-family:monospace;font-size:19px;font-weight:700;color:#a6cf46;">{pct}%</span>'
                   f'<span style="font-size:12.5px;color:#c8dd8f;"><b style="color:#e7f3cd;">{game}</b>'
                   f' &mdash; players recommend it &middot; {len(g)} reviews</span></div>')
    head = ('<div style="font-family:monospace;font-size:10.5px;letter-spacing:1.2px;'
            'text-transform:uppercase;color:#57b0e6;margin-bottom:10px;">Ask the reviews</div>')
    ans = ('<div style="font-size:13.5px;color:#dbe4ee;line-height:1.5;">From the reviews: '
           f'<mark style="background:#123049;color:#bfe2fb;padding:1px 6px;border-radius:5px;">{answer}</mark></div>')
    return _card(head + verdict + ans + _meter('answer confidence', score, '#57b0e6'))

def _router_html(pairs):
    head = ('<div style="font-family:monospace;font-size:10.5px;letter-spacing:1.2px;'
            'text-transform:uppercase;color:#a6cf46;margin-bottom:10px;">Review Router &middot; zero-shot tags</div>')
    rows = ''
    for i, (lab, sc) in enumerate(pairs):
        pct = int(round(sc * 100)); color = BAR_COLORS[i % len(BAR_COLORS)]
        rows += ('<div style="display:grid;grid-template-columns:140px 1fr 42px;align-items:center;'
                 'gap:10px;margin:7px 0;">'
                 f'<span style="font-size:13px;color:#dbe4ee;">{lab}</span>'
                 '<span style="height:7px;background:#0e1620;border:1px solid #243040;border-radius:99px;'
                 'display:block;overflow:hidden;">'
                 f'<span style="display:block;height:100%;width:{pct}%;background:{color};"></span></span>'
                 f'<span style="font-family:monospace;font-size:11.5px;color:#93a4b7;text-align:right;">{sc:.2f}</span></div>')
    return _card(head + rows)

def assistant(message, history=None, labels=DEFAULT_LABELS):
    if looks_like_question(message):
        game = detect_game(message)
        ctx = ' '.join(top_reviews(message, k=3, game=game))[:2000]
        a = qa(question=message, context=ctx)
        return _qa_html(game, a['answer'], a['score'])
    labs = [l.strip() for l in (labels or DEFAULT_LABELS).split(',') if l.strip()]
    r = router(message[:400], candidate_labels=labs, multi_label=True)
    return _router_html(list(zip(r['labels'], r['scores']))[:5])

# preview the cards right here (full HTML renders in the notebook, no Gradio needed)
display(HTML(assistant('Is Elden Ring worth it?')))
display(HTML(assistant('Gorgeous game but it ran like garbage at launch. Fixed now.')))

## 6 · The chat app — the whole class joins
A dark, Steam-styled chat. `launch(share=True)` prints a **public URL** (good ~72h) — put it on screen; everyone opens it on a laptop or phone.

- **Ask a question** about a game → QA answers, with a recommend-verdict.
- **Paste a review** → the Router tags it. Edit the **labels** box to invent your own categories.

**Your mission:** find one thing each tool gets *confidently wrong* — and name *why*.

In [ ]:
import gradio as gr

QUICKPICKS = ['Elden Ring', 'Sekiro', 'Cyberpunk 2077', 'Hades', "Baldur's Gate 3", 'Stardew Valley']

CSS = """
.gradio-container{background:#0c1118 !important;}
footer{display:none !important;}
#syp-header{display:flex;align-items:center;gap:14px;padding:12px 4px 8px;}
#syp-header .mark{width:38px;height:38px;border-radius:10px;background:#0e1a26;border:1px solid #2d3a4c;
  display:grid;place-items:center;color:#57b0e6;font-size:20px;}
#syp-header .ttl{font-size:17px;font-weight:650;color:#dbe4ee;}
#syp-header .sub{font-size:12px;color:#93a4b7;}
#syp-header .live{margin-left:auto;font-family:ui-monospace,monospace;font-size:11.5px;color:#93a4b7;
  background:#0e1620;border:1px solid #243040;padding:6px 11px;border-radius:999px;}
"""

theme = gr.themes.Base(primary_hue='blue', neutral_hue='slate').set(
    body_background_fill='#0c1118', block_background_fill='#151d27',
    block_border_color='#243040', body_text_color='#dbe4ee',
    input_background_fill='#0d141d', input_border_color='#2d3a4c',
    button_primary_background_fill='#57b0e6', button_primary_text_color='#052033',
)

HEADER = ('<div id="syp-header"><div class="mark">&#127918;</div>'
          '<div><div class="ttl">Should You Play It?</div>'
          '<div class="sub">Steam review assistant &middot; ISBA 2411</div></div>'
          f'<div class="live">&#9679; live &middot; {len(df):,} reviews &middot; {df.game.nunique()} games</div></div>')

with gr.Blocks(theme=theme, css=CSS, title='Should You Play It?') as demo:
    gr.HTML(HEADER)
    chat = gr.Chatbot(height=430, sanitize_html=False, type='messages', show_label=False)
    with gr.Row():
        picks = [gr.Button(g, size='sm') for g in QUICKPICKS]
    with gr.Row():
        msg  = gr.Textbox(placeholder='Ask about a game, or paste a review to tag...',
                          show_label=False, scale=5)
        send = gr.Button('Send', variant='primary', scale=1)
    labels = gr.Textbox(value=DEFAULT_LABELS, label='Router labels (invent your own!)')

    def respond(message, chat_history, labels_val):
        if not message.strip():
            return '', chat_history
        reply = assistant(message, labels=labels_val)
        chat_history = (chat_history or []) + [
            {'role': 'user', 'content': message},
            {'role': 'assistant', 'content': reply}]
        return '', chat_history

    send.click(respond, [msg, chat, labels], [msg, chat])
    msg.submit(respond, [msg, chat, labels], [msg, chat])
    for b, g in zip(picks, QUICKPICKS):
        b.click(lambda g=g: f'Is {g} worth it?', None, msg)

demo.launch(share=True)

### Fallback (if campus wi-fi blocks the public link)
Same brain, inline — no public URL. (Uses the default labels for tagging.)

In [ ]:
import ipywidgets as w
from IPython.display import display, HTML
box = w.Text(placeholder='Ask about a game, or paste a review...', layout=w.Layout(width='70%'))
send = w.Button(description='Send', button_style='primary'); out = w.Output()
def handle(_):
    if not box.value.strip(): return
    ans = assistant(box.value)
    with out: display(HTML(f'<div style="color:#93a4b7;font-size:12px;margin-top:8px;">you: '
                           f'{box.value}</div>' + ans))
    box.value = ''
send.on_click(handle)
display(w.HBox([box, send]), out)

---
## Take-home exercise
1. **Router:** pick 15 reviews and a label set of your own. Do the top tags match what the review is really about? Find two the Router gets wrong.
2. **QA:** ask five game questions. Find one it nails and one it fumbles — name the failure mode (wrong game retrieved / answer not in the text / yes-no question).
3. **Bonus:** compare the warm-up sentiment label to the `recommended` thumbs-up on 20 reviews — how often do they agree?

Hand in a short write-up: approach, what you found, three failure cases. *(Maps to the midterm rubric's Analysis & Insight.)*